In [1]:
!pip install -q sentence-transformers faiss-cpu transformers accelerate pypdf gradio

In [2]:
import os
import numpy as np
import faiss
import torch
import gradio as gr

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import pipeline

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=device
)

generator = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    device=0 if device == "cuda" else -1
)

print("AI models loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

... (3 outputs left out)

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'AXK1ForCausalLM', 'AXK2ForCausalLM', 'BailingHybridForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BeamForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CohereCompassForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV32ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCa

AI models loaded successfully.


In [16]:
from google.colab import files

uploaded = files.upload()

pdf_paths = list(uploaded.keys())

print("Uploaded files:", pdf_paths)

Saving Geneartive AI_1_1786948294297.pdf to Geneartive AI_1_1786948294297.pdf
Saving Geneartive AI.pdf to Geneartive AI (1).pdf
Uploaded files: ['Geneartive AI_1_1786948294297.pdf', 'Geneartive AI (1).pdf']


In [17]:
documents = []

for pdf_path in pdf_paths:
    reader = PdfReader(pdf_path)

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""

        if text.strip():
            documents.append({
                "text": text,
                "source": os.path.basename(pdf_path),
                "page": page_number
            })

print("Total pages with extracted text:", len(documents))

Total pages with extracted text: 44


In [18]:
def create_chunks(documents, chunk_size=1200, overlap=200):
    chunks = []

    for doc in documents:
        text = doc["text"]
        start = 0

        while start < len(text):
            end = start + chunk_size

            chunk_text = text[start:end].strip()

            if chunk_text:
                chunks.append({
                    "text": chunk_text,
                    "source": doc["source"],
                    "page": doc["page"]
                })

            if end >= len(text):
                break

            start = end - overlap

    return chunks


chunks = create_chunks(documents)

print("Total chunks:", len(chunks))
print(chunks[0] if chunks else "No readable text found.")

Total chunks: 84
{'text': 'B. Tech (Computer Science Business Systems) \nAcademic Year -2027-2028 (Semester –VII) \nProposed Structure Semester -VII  \n(Level 6 – B.Tech ) -Semester VII \n[CB4201T]:  Generative AI \nTeaching Scheme:  \nTH: 3 Hours/Week  \n \nCredit \nTH:3 \n \nExamination Scheme: \nIn Sem. Evaluation:20 Marks \nMid Sem. Exam      : 30Marks \nEnd Sem. Exam      : 50 Marks \nCourse Prerequisites: Linear Algebra, Computational Statistics[CB2104],Machine \nLearning[CB3101], Deep Learning[CB3109A] \nCourse Objective: \n• Understand the foundations and evolution of Generative AI systems. \n• Explain transformer architectures and modern Large Language Models. \n• Apply prompt engineering, reasoning, and context engineering techniques. \n• Design Retrieval-Augmented Generation (RAG) and Agentic AI applications. \n• Deploy and optimize LLM-based systems for enterprise use. \n• Analyze ethical, security, governance, and privacy challenges in Generative AI \nCourse Outcome: \nAft

In [19]:
if not chunks:
    raise ValueError(
        "No text extracted. Check whether your PDFs contain readable text."
    )

texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

embeddings = embeddings.astype("float32")

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Embedding shape: (84, 384)


In [20]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("Vectors indexed:", index.ntotal)

Vectors indexed: 84


In [21]:
def retrieve_chunks(question, top_k=4):
    query_vector = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    k = min(top_k, index.ntotal)

    scores, indices = index.search(query_vector, k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx >= 0:
            result = chunks[int(idx)].copy()
            result["score"] = float(score)
            results.append(result)

    return results

In [22]:
results = retrieve_chunks(
    "What are the rules for the end-semester examination?"
)

for result in results:
    print("Source:", result["source"])
    print("Page:", result["page"])
    print("Similarity:", round(result["score"], 3))
    print("Text:", result["text"][:300])
    print("-" * 50)

Source: Geneartive AI_1_1786948294297.pdf
Page: 4
Similarity: 0.287
Text: iments performed, submission of results of program 
in the form of report/journal, timely completion, attendance, understanding, efficient codes, 
punctuality and neatness. 
2) Practical/Oral examination shall be based on the practical performed in the lab.  
3) Lab assessment of 25 marks shall be b
--------------------------------------------------
Source: Geneartive AI (1).pdf
Page: 23
Similarity: 0.249
Text: s stage, you can identify the specific tasks or instructions 
the model should be capable of handling. This helps set the stage for designing 
appropriate prompts and planning for potential tuning strategies later on.
--------------------------------------------------
Source: Geneartive AI_1_1786948294297.pdf
Page: 4
Similarity: 0.242
Text: Final Year B. Tech (Computer Science Business Systems) 
Academic Year – 2026 -2027 Semester -VII  
 [CB2301L]: Generative AI Lab 
Teaching Scheme:  
Practical :  2 Hour

In [23]:
def answer_question(question):
    if not question or not question.strip():
        return "Please enter a question.", ""

    results = retrieve_chunks(question, top_k=4)

    if not results:
        return "No relevant information was retrieved.", ""

    context_parts = []

    for result in results:
        context_parts.append(
            f"[Source: {result['source']}, "
            f"Page: {result['page']}]\n{result['text']}"
        )

    context = "\n\n".join(context_parts)

    prompt = f"""
Answer the question using only the reference material below.
Do not invent university rules, dates, marks, or requirements.
If the answer is not stated in the material, say:
"The uploaded documents do not provide enough information
to answer this question."

Reference material:
{context}

Question: {question}

Answer:
"""

    output = generator(
        prompt,
        max_new_tokens=180,
        do_sample=False,
        truncation=True
    )

    answer = output[0]["generated_text"].strip()

    sources = "\n".join(
        f"- {r['source']}, page {r['page']}"
        for r in results
    )

    return answer, sources

In [24]:
question = "What topics are included in Unit 3?"

answer, sources = answer_question(question)

print("ANSWER:")
print(answer)

print("\nSOURCES:")
print(sources)

ANSWER:
Answer the question using only the reference material below.
Do not invent university rules, dates, marks, or requirements.
If the answer is not stated in the material, say:
"The uploaded documents do not provide enough information
to answer this question."

Reference material:
[Source: Geneartive AI_1_1786948294297.pdf, Page: 7]
to get structured, readable 
human outputs. 
CO3: Illustrate multimodal AI systems (Text, Image, Video). 
• PO3 (3): Designing generative pipelines for media requires creative system design (Stable 
Diffusion, GANs). 
• PO5 (3): Involves using state-of-the-art tools for ASR, TTS, and image generation. 
• PO10 (3): Multimodal outputs are inherently about communication—producing visual or 
audio content for healthcare, education, or marketing. 
CO4: Apply RAG and Agentic AI concepts for real-world problems. 
• PO1 (3): Requires mathematical knowledge of embeddings and vector search. 
• PO2 (3): Breaking down a university chatbot or enterprise automation 

In [25]:
def chat(question):
    answer, sources = answer_question(question)

    return f"{answer}\n\nSources:\n{sources}"


demo = gr.Interface(
    fn=chat,
    inputs=gr.Textbox(
        label="Ask the University Exam Assistant",
        placeholder="What is the passing requirement?",
        lines=2
    ),
    outputs=gr.Textbox(
        label="Answer and Sources",
        lines=10
    ),
    title="University Examination Assistant",
    description=(
        "Ask questions based on the syllabus and "
        "examination guidelines uploaded to the system."
    )
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://35d64c6c4f85c7c4fa.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
